In [0]:
%run ../functions/functions

In [0]:
# ==============================
# Lista dos datasets
# ==============================

datasets = [
    "EXP_2021", 
    "EXP_2021_MUN",
    "EXP_2022", 
    "EXP_2022_MUN",
    "EXP_2023", 
    "EXP_2023_MUN",
    "EXP_2024", 
    "EXP_2024_MUN",
    "EXP_2025", 
    "EXP_2025_MUN",
    "IMP_2021", 
    "IMP_2021_MUN",
    "IMP_2022", 
    "IMP_2022_MUN",
    "IMP_2023", 
    "IMP_2023_MUN",
    "IMP_2024", 
    "IMP_2024_MUN",
    "IMP_2025", 
    "IMP_2025_MUN",
    "ISIC_CUCI",
    "NBM",
    "NBM_NCM", 
    "NCM",
    "NCM_CGCE",
    "NCM_CUCI",
    "NCM_FAT_AGREG",
    "NCM_ISIC",
    "NCM_PPE",
    "NCM_PPI",
    "NCM_SH",
    "NCM_UNIDADE",
    "PAIS",
    "PAIS_BLOCO",
    "UF",
    "UF_MUN",
    "URF",
    "VIA"
]

In [0]:
dfs = load_all_datasets(datasets)

In [0]:
#print dos Schemas da bronze
for nome, df in dfs.items():
    print(nome)
    df.printSchema()
    print("-------------------------------------------------------------")

In [0]:
# ==============================
# Lista dos datasets de EXPORTACAO UF
# ==============================

# 1. Configurações específicas para o dataset
exp_cast_config = {
    "CO_ANO": "int",
    "CO_MES": "int",
    "CO_NCM": "long",
    "CO_UNID": "int",
    "CO_PAIS": "int",
    "CO_VIA": "int",
    "CO_URF": "int",
    "QT_ESTAT": "double",
    "KG_LIQUIDO": "double",
    "VL_FOB": "double"
}

exp_business_keys = [
    "CO_ANO", "CO_MES", "CO_NCM", "CO_UNID", 
    "CO_PAIS", "SG_UF_NCM", "CO_VIA", "CO_URF"
]


# 2. Execução do pipeline
datasets = ["EXP_2021", "EXP_2022","EXP_2023", "EXP_2024", "EXP_2025"]
dfs = load_all_datasets(datasets)

df_silver_exp = process_silver_layer(
    dfs_dict=dfs, 
    cast_config=exp_cast_config, 
    business_keys=exp_business_keys, 
    sk_name="SK_EXPORTACAO"
)


# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_exp, 
    table_name="EXP_CONSOLIDADA", 
    primary_keys=["SK_EXPORTACAO"]
)

In [0]:
# ==============================
# Lista dos datasets de EXPORTACAO MUNICIPIO
# ==============================

# 1. Configurações específicas para o dataset
exp_mun_cast_config = {
    "CO_ANO": "int",
    "CO_MES": "int",
    "SH4": "long",
    "CO_PAIS": "int",
    "SG_UF_MUN": "int",
    "CO_MUN": "int",
    "KG_LIQUIDO": "double",
    "VL_FOB": "double",
}

exp_business_keys = [
    "CO_ANO", "CO_MES", "SH4", "CO_PAIS", 
    "SG_UF_MUN", "CO_MUN"
]


# 2. Execução do pipeline
datasets = ["EXP_2021", "EXP_2022"]
dfs = load_all_datasets(datasets)

df_silver_exp = process_silver_layer(
    dfs_dict=dfs, 
    cast_config=exp_mun_cast_config, 
    business_keys=exp_business_keys, 
    sk_name="SK_EXPORTACAO"
)


# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_exp, 
    table_name="EXP_CONSOLIDADA", 
    primary_keys=["SK_EXPORTACAO"]
)

In [0]:
dfs["EXP_2021_MUN"].display(10)

In [0]:
# 1. Configurações específicas para o dataset de VIA
via_cast_config = {
    "CO_VIA": "int",
    "NO_VIA": "string"
}

via_business_keys = [
    "CO_VIA"
]


In [0]:
# 2. Execução do pipeline VIA
datasets = ["VIA"]
dfs_via = load_all_datasets(datasets)

df_silver_via = process_silver_layer(
    dfs_dict=dfs_via, 
    cast_config=via_cast_config, 
    business_keys=via_business_keys, 
    sk_name="SK_VIA"
)

df_silver_via.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_via, 
    table_name="VIA_CONSOLIDADA", 
    primary_keys=["SK_VIA"]
)

In [0]:
#validacao do union
dfs["EXP_2021"].count() + dfs["EXP_2022"].count() - df_silver_exp.count()

In [0]:
df_silver_exp.printSchema()

In [0]:
# Definindo o caminho
silver_path = "abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/EXP_CONSOLIDADA/"

write_silver(df, dataset)

# Lendo os dados
df_silver = spark.read.format("delta").load(silver_path)

# Verificando o resultado
#df_silver.count()

In [0]:
df_silver.count()
#2976163

# NCM SH

In [0]:


dfs['NCM_SH'].printSchema()
dfs['NCM_SH'].display()


In [0]:
# 1. Configurações específicas para o dataset de NCM_SH
ncm_sh_cast_config = {
    "CO_SH6": "int",
    "NO_SH6_POR": "string",
    "NO_SH6_ESP": "string",
    "NO_SH6_ING": "string",
    "CO_SH4": "int",
    "NO_SH4_POR": "string",
    "NO_SH4_ESP": "string",
    "NO_SH4_ING": "string",
    "CO_SH2": "int",
    "NO_SH2_POR": "string",
    "NO_SH2_ESP": "string",
    "NO_SH2_ING": "string",
    "CO_NCM_SECROM": "string",
    "NO_SEC_POR": "string",
    "NO_SEC_ESP": "string",
    "NO_SEC_ING": "string"
}
 
ncm_sh_business_keys = [
    "CO_SH6","CO_SH4","CO_SH2"
]





In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_SH"]
dfs_ncm_sh = load_all_datasets(datasets)
 
df_silver_ncm_sh = process_silver_layer(
    dfs_dict=dfs_ncm_sh,
    cast_config=ncm_sh_cast_config,
    business_keys=ncm_sh_business_keys,
    sk_name="SK_NCM_SH"
)
 
# df_silver_pais.display()


colunas_incorretas = ['NO_SH6_POR', 'NO_SH6_ESP','NO_SH6_ING','NO_SH4_POR','NO_SH4_ESP','NO_SH4_ING','NO_SH2_POR','NO_SH2_ESP','NO_SH2_ING','CO_NCM_SECROM','NO_SEC_POR','NO_SEC_ESP','NO_SEC_ING']
 
df_silver_ncm_sh = tratamento(df_silver_ncm_sh, colunas_incorretas)

df_silver_ncm_sh.display()



In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_sh ,
    table_name="NCM_SH_CONSOLIDADA",
    primary_keys=["SK_NCM_SH"]
)

# ISIS_CUCI

In [0]:
dfs['ISIC_CUCI'].printSchema()
dfs['ISIC_CUCI'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_SH
isis_cuci_cast_config = {
    "CO_ISIC_SECAO": "string",
    "NO_ISIC_SECAO": "string",
    "CO_CUCI_GRUPO": "string",
    "NO_CUCI_GRUPO": "string"
}
 
isis_cuci_business_keys = [
    "CO_ISIC_SECAO", "NO_ISIC_SECAO","CO_CUCI_GRUPO"
]





In [0]:
# 2. Execução do pipeline VIA
datasets = ["ISIC_CUCI"]
dfs_isis_cuci = load_all_datasets(datasets)
 
df_silver_isis_cuci = process_silver_layer(
    dfs_dict=dfs_isis_cuci,
    cast_config=isis_cuci_cast_config,
    business_keys=isis_cuci_business_keys,
    sk_name="SK_ISIS_CUCI"
)
 
# df_silver_pais.display()


colunas_incorretas = ['NO_ISIC_SECAO', 'NO_CUCI_GRUPO']
 
df_silver_isis_cuci  = tratamento(df_silver_isis_cuci , colunas_incorretas)

df_silver_isis_cuci .display()



In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_isis_cuci,
    table_name="ISIS_CUCI_CONSOLIDADA",
    primary_keys=["SK_ISIS_CUCI"]
)

# NBM_NCM

In [0]:
dfs['NBM_NCM'].printSchema()
dfs['NBM_NCM'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_SH
nbm_ncm_cast_config = {
    "CO_NBM": "long",
    "CO_NCM": "long"
}
 
nbm_ncm_business_keys = [
    "CO_NBM"
]




In [0]:
# 2. Execução do pipeline VIA
datasets = ["NBM_NCM"]
dfs_nbm_ncm = load_all_datasets(datasets)
 
df_silver_nbm_ncm = process_silver_layer(
    dfs_dict=dfs_nbm_ncm,
    cast_config=nbm_ncm_cast_config,
    business_keys=nbm_ncm_business_keys,
    sk_name="SK_NBM_NCM"
)

df_silver_nbm_ncm.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_nbm_ncm,
    table_name="NBM_NCM_CONSOLIDADA",
    primary_keys=["SK_NBM_NCM"]
)

# NCM_CGCE

In [0]:
dfs['NCM_CGCE'].printSchema()
dfs['NCM_CGCE'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE
ncm_cgce_cast_config = {
    "CO_CGCE_N3": "int",
    "NO_CGCE_N3": "string",
    "NO_CGCE_N3_ING": "string",
    "NO_CGCE_N3_ESP": "string",
    "CO_CGCE_N2": "int",
    "NO_CGCE_N2": "string",
    "NO_CGCE_N2_ING": "string",
    "NO_CGCE_N2_ESP": "string",
    "CO_CGCE_N1": "int",
    "NO_CGCE_N1": "string",
    "NO_CGCE_N1_ING": "string",
    "NO_CGCE_N1_ESP": "string"

}
 
ncm_cgce_business_keys = [
    "CO_CGCE_N3","CO_CGCE_N2","CO_CGCE_N1"
]




In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_CGCE"]
dfs_ncm_cgce = load_all_datasets(datasets)
 
df_silver_ncm_cgce= process_silver_layer(
    dfs_dict=dfs_ncm_cgce,
    cast_config=ncm_cgce_cast_config,
    business_keys=ncm_cgce_business_keys,
    sk_name="SK_NCM_CGCE"
)

colunas_incorretas = ['NO_CGCE_N3','NO_CGCE_N3_ING','NO_CGCE_N3_ESP','NO_CGCE_N2','NO_CGCE_N2_ING','NO_CGCE_N2_ESP','NO_CGCE_N1','NO_CGCE_N1_ING','NO_CGCE_N1_ESP']

df_silver_ncm_cgce  = tratamento(df_silver_ncm_cgce , colunas_incorretas)

df_silver_ncm_cgce .display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_cgce,
    table_name="NCM_CGCE_CONSOLIDADA",
    primary_keys=["SK_NCM_CGCE"]
)

# NCM_CUCI


In [0]:
dfs['NCM_CUCI'].printSchema()
dfs['NCM_CUCI'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE
ncm_cuci_cast_config = {
    "CO_CUCI_ITEM": "string",
    "NO_CUCI_ITEM": "string",
    "CO_CUCI_SUB": "int",
    "NO_CUCI_SUB": "string",
    "CO_CUCI_GRUPO": "int",
    "NO_CUCI_GRUPO": "string",
    "CO_CUCI_DIVISAO": "string",
    "NO_CUCI_DIVISAO": "string",
    "CO_CUCI_SEC": "int",
    "NO_CUCI_SEC": "string",
}
 
ncm_cuci_business_keys = [
    "CO_CGCE_N3","CO_CGCE_N2","CO_CGCE_N1"
]


In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_CUCI"]
dfs_ncm_cuci = load_all_datasets(datasets)
 
df_silver_ncm_cuci= process_silver_layer(
    dfs_dict=dfs_ncm_cgce,
    cast_config=ncm_cgce_cast_config,
    business_keys=ncm_cgce_business_keys,
    sk_name="SK_NCM_CGCE"
)

colunas_incorretas = ['NO_CUCI_ITEM','NO_CUCI_SUB','NO_CUCI_GRUPO','NO_CUCI_DIVISAO','NO_CUCI_SEC']

df_silver_ncm_cuci  = tratamento(df_silver_ncm_cuci, colunas_incorretas)

df_silver_ncm_cuci.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_cuci,
    table_name="NCM_CUCI_CONSOLIDADA",
    primary_keys=["SK_NCM_CUCI"]
)

# NCM_FAT_AGREG

In [0]:
dfs['NCM_FAT_AGREG'].printSchema()
dfs['NCM_FAT_AGREG'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE
ncm_fat_agreg_cast_config = {
    "CO_FAT_AGREG": "int",
    "NO_FAT_AGREG": "string",
    "NO_FAT_AGREG_GP": "string"
}
 
ncm_fat_agreg_business_keys = [
    "CO_FAT_AGREG"
]


In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_FAT_AGREG"]
dfs_ncm_fat_agreg = load_all_datasets(datasets)
 
df_silver_ncm_fat_agreg = process_silver_layer(
    dfs_dict=dfs_ncm_fat_agreg ,
    cast_config=ncm_fat_agreg _cast_config,
    business_keys=ncm_fat_agreg _business_keys,
    sk_name="SK_NCM_FAT_AGREG"
)

colunas_incorretas = ['NO_CUCI_ITEM','NO_CUCI_SUB','NO_CUCI_GRUPO','NO_CUCI_DIVISAO','NO_CUCI_SEC']

df_silver_ncm_fat_agreg  = tratamento(df_silver_ncm_fat_agreg, colunas_incorretas)

df_silver_ncm_fat_agreg.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_fat_agreg,
    table_name="NCM_FAT_AGREG_CONSOLIDADA",
    primary_keys=["SK_NCM_FAT_AGREG"]
)

# NCM_ISIC

In [0]:
dfs['NCM_ISIC'].printSchema()
dfs['NCM_ISIC'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE
ncm_isic_cast_config = {
    "CO_ISIC_CLASSE": "int",
    "NO_ISIC_CLASSE": "string",
    "NO_ISIC_CLASSE_ING": "string",
    "NO_ISIC_CLASSE_ESP": "string",
    "CO_ISIC_GRUPO": "int",
    "NO_ISIC_GRUPO": "string",
    "NO_ISIC_GRUPO_ING": "string",
    "NO_ISIC_GRUPO_ESP": "string",
    "CO_ISIC_DIVISAO": "int",
    "NO_ISIC_DIVISAO_ING": "string",
    "NO_ISIC_DIVISAO_ESP": "string",
    "CO_ISIC_SECAO": "string",
    "NO_ISIC_SECAO": "string",
    "NO_ISIC_SECAO_ING": "string",
    "NO_ISIC_SECAO_ESP": "string",

}
 
ncm_isic_business_keys = [
    "CO_ISIC_CLASSE","CO_ISIC_GRUPO","CO_ISIC_DIVISAO","CO_ISIC_SECAO"
]




In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_ISIC"]
dfs_ncm_isic = load_all_datasets(datasets)
 
df_silver_ncm_isic  = process_silver_layer(
    dfs_dict=dfs_ncm_isic  ,
    cast_config=ncm_isic_cast_config,
    business_keys=ncm_isic_business_keys,
    sk_name="SK_NCM_ISIC"
)

colunas_incorretas = ['NO_ISIC_CLASSE','NO_ISIC_CLASSE_ING','NO_ISIC_CLASSE_ESP','NO_ISIC_GRUPO','NO_ISIC_GRUPO_ING','NO_ISIC_GRUPO_ESP','CO_ISIC_DIVISAO','NO_ISIC_DIVISAO_ING','NO_ISIC_DIVISAO_ESP','NO_ISIC_SECAO','NO_ISIC_SECAO_ING','NO_ISIC_SECAO_ESP']


df_silver_ncm_isic  = tratamento(df_silver_ncm_isic, colunas_incorretas)

df_silver_ncm_isic.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_isic,
    table_name="NCM_ISIC",
    primary_keys=["SK_NCM_ISIC"]
)

# NCM_PPE

In [0]:
dfs['NCM_PPE'].printSchema()
dfs['NCM_PPE'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE
ncm_ppe_cast_config = {
    "CO_PPE": "int",
    "NO_PPE": "string",
    "NO_PPE_MIN": "string",
    "NO_PPE_ING": "string"

}
 
ncm_ppe_business_keys = [
    "CO_PPE"
]

In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_PPE"]
dfs_ncm_ppe = load_all_datasets(datasets)
 
df_silver_ncm_ppe  = process_silver_layer(
    dfs_dict=dfs_ncm_ppe  ,
    cast_config=ncm_ppe_cast_config,
    business_keys=ncm_ppe_business_keys,
    sk_name="SK_NCM_PPE"
)

colunas_incorretas = ['NO_PPE','NO_PPE_MIN','NO_PPE_ING']

df_silver_ncm_ppe  = tratamento(df_silver_ncm_ppe, colunas_incorretas)

df_silver_ncm_ppe.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_ppe,
    table_name="NCM_PPE",
    primary_keys=["SK_NCM_PPE"]
)

# NCM_PPI 

In [0]:
dfs['NCM_PPI'].printSchema()
dfs['NCM_PPI'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE
ncm_ppi_cast_config = {
    "CO_PPI": "int",
    "NO_PPI": "string",
    "NO_PPI_MIN": "string",
    "NO_PPI_ING": "string"

}
 
ncm_ppi_business_keys = [
    "CO_PPI"
]

In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_PPI"]
dfs_ncm_ppi = load_all_datasets(datasets)
 
df_silver_ncm_ppi = process_silver_layer(
    dfs_dict=dfs_ncm_ppi ,
    cast_config=ncm_ppi_cast_config,
    business_keys=ncm_ppi_business_keys,
    sk_name="SK_NCM_PPI"
)

colunas_incorretas = ['NO_PPI','NO_PPI_MIN','NO_PPI_ING']

df_silver_ncm_ppi  = tratamento(df_silver_ncm_ppi, colunas_incorretas)

df_silver_ncm_ppi.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_ppi,
    table_name="NCM_PPI",
    primary_keys=["SK_NCM_PPI"]
)

# NCM_UNIDADE

In [0]:
dfs['NCM_UNIDADE'].printSchema()
dfs['NCM_UNIDADE'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE
ncm_unidade_cast_config = {
    "CO_UNID": "int",
    "NO_UNID": "string",
    "SG_UNID": "string"
}
 
ncm_unidade_business_keys = [
    "CO_UNID"
]

In [0]:
# 2. Execução do pipeline VIA
datasets = ["NCM_UNIDADE"]
dfs_ncm_unidade = load_all_datasets(datasets)
 
df_silver_ncm_unidade = process_silver_layer(
    dfs_dict=dfs_ncm_unidade,
    cast_config=ncm_unidade_cast_config,
    business_keys=ncm_unidade_business_keys,
    sk_name="SK_NCM_UNIDADE"
)

colunas_incorretas = ['NO_UNID','SG_UNID']

df_silver_ncm_unidade  = tratamento(df_silver_ncm_unidade, colunas_incorretas)

df_silver_ncm_unidade.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_ncm_unidade,
    table_name="NCM_UNIDADE",
    primary_keys=["SK_NCM_UNIDADE"]
)

# PAIS

In [0]:
dfs['PAIS'].printSchema()
dfs['PAIS'].display()

In [0]:
# 1. Configurações específicas para o dataset de NCM_CGCE

pais_cast_config = {
    "CO_PAIS": "int",
    "CO_PAIS_ISON3": "int",
    "CO_PAIS_ISOA3": "string",
    "NO_PAIS": "string",
    "NO_PAIS_ING": "string",
    "NO_PAIS_ESP": "string"
    
}
 
pais_business_keys = [
    "CO_PAIS"
]



In [0]:
# 2. Execução do pipeline VIA
datasets = ["PAIS"]
dfs_pais = load_all_datasets(datasets)
 
df_silver_pais = process_silver_layer(
    dfs_dict=dfs_pais,
    cast_config=pais_cast_config,
    business_keys=pais_business_keys,
    sk_name="SK_PAIS
)

colunas_incorretas = ['CO_PAIS_ISOA3','NO_PAIS','NO_PAIS_ING','NO_PAIS_ESP']

df_silver_pais  = tratamento(df_silver_pais, colunas_incorretas)

df_silver_pais.display()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_pais,
    table_name="PAIS",
    primary_keys=["SK_PAIS"]
)